<a href="https://colab.research.google.com/github/Tejas-Acharya-C/api-data-engineering-pipeline/blob/main/api_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# API → Data Engineering Pipeline

A hands-on data engineering project covering:

- API data extraction
- JSON processing
- Data cleaning
- Data validation
- Parquet
- DuckDB
- SQL analytics
- Incremental processing
- Python pipeline development

## 1. Extract Data from API

We will make an HTTP GET request to a public API and inspect the JSON response.

In [1]:
import requests

In [2]:
url = "https://jsonplaceholder.typicode.com/users"

In [3]:
response = requests.get(url)

In [4]:
print(response.status_code)

200


In [5]:
data = response.json()

In [6]:
print(type(data))

<class 'list'>


In [7]:
print(len(data))

10


In [8]:
print(data[0])

{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


In [9]:
print(response.status_code)
print(type(data))
print(len(data))
print(data[0])

200
<class 'list'>
10
{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


## 2. Store Raw API Data

We save the original API response before performing any transformations.

In [10]:
import os

os.makedirs("data/raw", exist_ok=True)

In [11]:
import json

with open("data/raw/users.json", "w") as f:
    json.dump(data, f, indent=2)

In [12]:
os.listdir("data/raw")

['users.json']

## 3. Convert JSON to DataFrame

Convert the nested JSON response into a tabular structure using Pandas.

In [13]:
import pandas as pd

In [14]:
df = pd.DataFrame(data)

In [15]:
df.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


In [16]:
df = pd.json_normalize(data)

In [17]:
df.head()

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


In [18]:
df.columns.tolist()

['id',
 'name',
 'username',
 'email',
 'phone',
 'website',
 'address.street',
 'address.suite',
 'address.city',
 'address.zipcode',
 'address.geo.lat',
 'address.geo.lng',
 'company.name',
 'company.catchPhrase',
 'company.bs']

In [20]:
df.shape

(10, 15)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   10 non-null     int64 
 1   name                 10 non-null     object
 2   username             10 non-null     object
 3   email                10 non-null     object
 4   phone                10 non-null     object
 5   website              10 non-null     object
 6   address.street       10 non-null     object
 7   address.suite        10 non-null     object
 8   address.city         10 non-null     object
 9   address.zipcode      10 non-null     object
 10  address.geo.lat      10 non-null     object
 11  address.geo.lng      10 non-null     object
 12  company.name         10 non-null     object
 13  company.catchPhrase  10 non-null     object
 14  company.bs           10 non-null     object
dtypes: int64(1), object(14)
memory usage: 1.3+ KB


In [22]:
df.head(3)

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications


In [23]:
df.tail(3)

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,586.493.6943 x140,jacynthe.com,Ellsworth Summit,Suite 729,Aliyaview,45169,-14.3990,-120.7677,Abernathy Group,Implemented secondary concept,e-enable extensible e-tailers
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project,aggregate real-time technologies
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,024-648-3804,ambrose.net,Kattie Turnpike,Suite 198,Lebsackbury,31428-2261,-38.2386,57.2232,Hoeger LLC,Centralized empowering task-force,target end-to-end models


In [24]:
df.shape
df.columns.tolist()
df.head(3)

,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications


In [25]:
df.shape

(10, 15)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   10 non-null     int64 
 1   name                 10 non-null     object
 2   username             10 non-null     object
 3   email                10 non-null     object
 4   phone                10 non-null     object
 5   website              10 non-null     object
 6   address.street       10 non-null     object
 7   address.suite        10 non-null     object
 8   address.city         10 non-null     object
 9   address.zipcode      10 non-null     object
 10  address.geo.lat      10 non-null     object
 11  address.geo.lng      10 non-null     object
 12  company.name         10 non-null     object
 13  company.catchPhrase  10 non-null     object
 14  company.bs           10 non-null     object
dtypes: int64(1), object(14)
memory usage: 1.3+ KB


In [27]:
df.isnull().sum()

,0
id,0
name,0
username,0
email,0
phone,0
website,0
address.street,0
address.suite,0
address.city,0
address.zipcode,0


In [28]:
df.duplicated().sum()

np.int64(0)

In [29]:
df["id"].duplicated().sum()

np.int64(0)

In [30]:
df["id"].is_unique

True

In [31]:
df.nunique()

,0
id,10
name,10
username,10
email,10
phone,10
website,10
address.street,10
address.suite,10
address.city,10
address.zipcode,10


In [34]:
df[["address.geo.lat", "address.geo.lng"]].head()

,address.geo.lat,address.geo.lng
0,-37.3159,81.1496
1,-43.9509,-34.4618
2,-68.6102,-47.0653
3,29.4572,-164.2990
4,-31.8129,62.5342


In [33]:
df[["address.geo.lat", "address.geo.lng"]].dtypes

,0
address.geo.lat,object
address.geo.lng,object


In [35]:
df.shape
df.info()
df.isnull().sum()
df.duplicated().sum()
df["id"].duplicated().sum()
df[["address.geo.lat", "address.geo.lng"]].dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   10 non-null     int64 
 1   name                 10 non-null     object
 2   username             10 non-null     object
 3   email                10 non-null     object
 4   phone                10 non-null     object
 5   website              10 non-null     object
 6   address.street       10 non-null     object
 7   address.suite        10 non-null     object
 8   address.city         10 non-null     object
 9   address.zipcode      10 non-null     object
 10  address.geo.lat      10 non-null     object
 11  address.geo.lng      10 non-null     object
 12  company.name         10 non-null     object
 13  company.catchPhrase  10 non-null     object
 14  company.bs           10 non-null     object
dtypes: int64(1), object(14)
memory usage: 1.3+ KB


,0
address.geo.lat,object
address.geo.lng,object


In [36]:
df = df.rename(columns={
    "address.street": "street",
    "address.suite": "suite",
    "address.city": "city",
    "address.zipcode": "zipcode",
    "address.geo.lat": "latitude",
    "address.geo.lng": "longitude",
    "company.name": "company_name",
    "company.catchPhrase": "company_catchphrase",
    "company.bs": "company_bs"
})

In [37]:
df.columns.tolist()

['id',
 'name',
 'username',
 'email',
 'phone',
 'website',
 'street',
 'suite',
 'city',
 'zipcode',
 'latitude',
 'longitude',
 'company_name',
 'company_catchphrase',
 'company_bs']

In [38]:
df[["latitude", "longitude"]].dtypes

,0
latitude,object
longitude,object


In [39]:
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])

In [40]:
df[["latitude", "longitude"]].dtypes

,0
latitude,float64
longitude,float64


In [41]:
df[["latitude", "longitude"]].head()

,latitude,longitude
0,-37.3159,81.1496
1,-43.9509,-34.4618
2,-68.6102,-47.0653
3,29.4572,-164.2990
4,-31.8129,62.5342


In [42]:
df[["latitude", "longitude"]].dtypes

,0
latitude,float64
longitude,float64


In [43]:
df.columns.tolist()
df[["latitude", "longitude"]].dtypes

,0
latitude,float64
longitude,float64


In [44]:
df["latitude"].min(), df["latitude"].max()

(-71.4197, 29.4572)

In [45]:
df["longitude"].min(), df["longitude"].max()

(-168.8889, 81.1496)

In [46]:
assert df["latitude"].between(-90, 90).all()
assert df["longitude"].between(-180, 180).all()

In [47]:
assert df["latitude"].between(-90, 90).all()

In [48]:
def validate_users(df):
    assert df["id"].notna().all(), "Missing user IDs"
    assert df["id"].is_unique, "Duplicate user IDs"

    assert df["email"].notna().all(), "Missing emails"

    assert df["latitude"].between(-90, 90).all(), "Invalid latitude"
    assert df["longitude"].between(-180, 180).all(), "Invalid longitude"

    return True

In [49]:
validate_users(df)

True

In [50]:
test_df = df.copy()

In [51]:
test_df.loc[0, "latitude"] = 200

In [52]:
validate_users(test_df)

AssertionError: Invalid latitude

In [53]:
validate_users(df)

True

In [54]:
os.makedirs("data/processed", exist_ok=True)

In [55]:
os.listdir("data")

['processed', 'raw']

In [56]:
df.to_parquet(
    "data/processed/users.parquet",
    index=False
)

In [57]:
os.listdir("data/processed")

['users.parquet']

In [58]:
processed_df = pd.read_parquet(
    "data/processed/users.parquet"
)

In [59]:
processed_df.head()

,id,name,username,email,phone,website,street,suite,city,zipcode,latitude,longitude,company_name,company_catchphrase,company_bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


In [60]:
import os

json_size = os.path.getsize("data/raw/users.json")
parquet_size = os.path.getsize("data/processed/users.parquet")

print("JSON:", json_size, "bytes")
print("Parquet:", parquet_size, "bytes")

JSON: 5645 bytes
Parquet: 11240 bytes


In [61]:
processed_df.shape

(10, 15)

In [62]:
processed_df.dtypes

,0
id,int64
name,object
username,object
email,object
phone,object
website,object
street,object
suite,object
city,object
zipcode,object


In [63]:
validate_users(processed_df)

True

In [66]:
os.makedirs("data/processed", exist_ok=True)

In [67]:
df.to_parquet(
    "data/processed/users.parquet",
    index=False
)

In [68]:
processed_df = pd.read_parquet(
    "data/processed/users.parquet"
)

In [69]:
processed_df.shape
processed_df.dtypes
validate_users(processed_df)

True

In [70]:
!pip install -q duckdb

In [71]:
import duckdb

In [72]:
con = duckdb.connect()

In [73]:
result = con.execute("""
    SELECT *
    FROM 'data/processed/users.parquet'
""").fetchdf()

In [74]:
result.head()

,id,name,username,email,phone,website,street,suite,city,zipcode,latitude,longitude,company_name,company_catchphrase,company_bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


In [75]:
con.execute("""
    SELECT
        id,
        name,
        city
    FROM 'data/processed/users.parquet'
""").fetchdf()

,id,name,city
0,1,Leanne Graham,Gwenborough
1,2,Ervin Howell,Wisokyburgh
2,3,Clementine Bauch,McKenziehaven
3,4,Patricia Lebsack,South Elvis
4,5,Chelsey Dietrich,Roscoeview
5,6,Mrs. Dennis Schulist,South Christy
6,7,Kurtis Weissnat,Howemouth
7,8,Nicholas Runolfsdottir V,Aliyaview
8,9,Glenna Reichert,Bartholomebury
9,10,Clementina DuBuque,Lebsackbury


In [76]:
con.execute("""
    SELECT
        id,
        name,
        city
    FROM 'data/processed/users.parquet'
    WHERE city = 'Gwenborough'
""").fetchdf()

,id,name,city
0,1,Leanne Graham,Gwenborough


In [77]:
con.execute("""
    SELECT
        id,
        name,
        city
    FROM 'data/processed/users.parquet'
    ORDER BY name
""").fetchdf()

,id,name,city
0,5,Chelsey Dietrich,Roscoeview
1,10,Clementina DuBuque,Lebsackbury
2,3,Clementine Bauch,McKenziehaven
3,2,Ervin Howell,Wisokyburgh
4,9,Glenna Reichert,Bartholomebury
5,7,Kurtis Weissnat,Howemouth
6,1,Leanne Graham,Gwenborough
7,6,Mrs. Dennis Schulist,South Christy
8,8,Nicholas Runolfsdottir V,Aliyaview
9,4,Patricia Lebsack,South Elvis


In [78]:
con.execute("""
    SELECT COUNT(*) AS total_users
    FROM 'data/processed/users.parquet'
""").fetchdf()

,total_users
0,10


In [79]:
con.execute("""
    SELECT
        city,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY city
""").fetchdf()

,city,user_count
0,Wisokyburgh,1
1,Howemouth,1
2,Aliyaview,1
3,Bartholomebury,1
4,Lebsackbury,1
5,Gwenborough,1
6,McKenziehaven,1
7,South Elvis,1
8,Roscoeview,1
9,South Christy,1


In [80]:
con.execute("""
    SELECT
        city,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY city
    ORDER BY user_count DESC
""").fetchdf()

,city,user_count
0,Gwenborough,1
1,McKenziehaven,1
2,South Elvis,1
3,Roscoeview,1
4,South Christy,1
5,Wisokyburgh,1
6,Howemouth,1
7,Aliyaview,1
8,Bartholomebury,1
9,Lebsackbury,1


In [81]:
con.execute("""
    SELECT
        company_name,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY company_name
    ORDER BY user_count DESC
""").fetchdf()

,company_name,user_count
0,Romaguera-Jacobson,1
1,Robel-Corkery,1
2,Considine-Lockman,1
3,Johns Group,1
4,Abernathy Group,1
5,Yost and Sons,1
6,Hoeger LLC,1
7,Romaguera-Crona,1
8,Deckow-Crist,1
9,Keebler LLC,1


In [87]:
con.execute("""
    SELECT
        company_name,
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    GROUP BY company_name
    ORDER BY user_count DESC
""").fetchdf()

,company_name,user_count
0,Romaguera-Jacobson,1
1,Robel-Corkery,1
2,Considine-Lockman,1
3,Johns Group,1
4,Romaguera-Crona,1
5,Deckow-Crist,1
6,Keebler LLC,1
7,Abernathy Group,1
8,Yost and Sons,1
9,Hoeger LLC,1


In [89]:
con.execute("""
    SELECT
        COUNT(*) AS user_count
    FROM 'data/processed/users.parquet'
    WHERE NAME LIKE '%a%'
""").fetchdf()

,user_count
0,7


In [90]:
con.execute("""
    SELECT
        name,email
    FROM 'data/processed/users.parquet'
    WHERE CITY LIKE '%view%'
""").fetchdf()

,name,email
0,Chelsey Dietrich,Lucio_Hettinger@annie.ca
1,Nicholas Runolfsdottir V,Sherwood@rosamond.me
